# Get To Know A Dataset: MemProtMD

This notebook is a guided tour of the [MemProtMD](https://registry.opendata.aws/memprotmd) dataset: coarse-grained molecular dynamics simulations of every membrane protein structure in the Protein Data Bank, each self-assembled into a lipid bilayer. More usage examples, tutorials and documentation for this dataset and others are on the [Registry of Open Data on AWS](https://registry.opendata.aws/).

Full documentation for the dataset: [README](https://github.com/stansfeldlab/stansfeldlab.github.io/blob/main/open-data/memprotmd/README.md). Website: https://stansfeldlab.github.io/memprotmd/

### Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.

The bucket has four top-level prefixes, plus a README, a licence and a manifest of every object:

1. **`catalog/`**: the place to start. `entries.tsv` has one row per simulation (title, method, resolution, organism, UniProt, bilayer thickness, classification). `residues.parquet` has one row per residue of every simulation, with its depth in the membrane and its contacts with lipids and water.
2. **`simulations/<pdb>_default_dppc/`**: one prefix per simulation, named after the PDB ID (e.g. `1a0s_default_dppc/`). It holds rendered `images/`, snapshots in `files/structures/`, per-residue analysis in `files/contacts/`, GROMACS inputs in `files/run/`, and analysis results as `*.latest.yml`.
3. **`databases/`**: images for each entry and for groups of entries (MemProtMD, TCDB, mpstruc and Pfam), thickness histograms per group, and the pipeline status of every PDB code considered.
4. **`ensembles/statistics/`**: statistics across all simulations.

PDB IDs are lower case. Older entries use the four-character code (`1a0s`); newer ones use the PDB's extended ID (`pdb_000010db`).

In [ ]:
# This notebook requires the following additional libraries
# (install with your preferred tool, e.g. pip or conda):
#
# boto3 >= 1.34
# pandas >= 2.1
# pyarrow >= 15
# pyyaml >= 6.0
# matplotlib >= 3.8
# pillow >= 10

import io

import boto3
import matplotlib.pyplot as plt
import pandas as pd
import pyarrow.dataset as ds
import pyarrow.fs
import yaml
from botocore import UNSIGNED
from botocore.config import Config
from PIL import Image

First, we set where the dataset lives and make an S3 client. The bucket is public, so requests do not need to be signed and no AWS account is needed.

In [ ]:
BUCKET = "memprotmd"      # draft: final bucket name to be confirmed
REGION = "us-west-2"      # draft: final region to be confirmed

s3 = boto3.client("s3", region_name=REGION, config=Config(signature_version=UNSIGNED))

def read(key):
    """The bytes of one object in the bucket."""
    return s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()

# The top-level prefixes and files
listing = s3.list_objects_v2(Bucket=BUCKET, Delimiter="/")
for p in listing.get("CommonPrefixes", []):
    print(p["Prefix"])
for o in listing.get("Contents", []):
    print(o["Key"])

Each simulation has its own prefix. Here are the files for 1A0S, the sucrose-specific porin ScrY, a β-barrel from the outer membrane of *Salmonella*:

In [ ]:
PDB = "1a0s"
prefix = f"simulations/{PDB}_default_dppc/"
for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=prefix):
    for o in page.get("Contents", []):
        print(f'{o["Size"]:>12,}  {o["Key"][len(prefix):]}')

### Q: What data formats are present in your dataset? What kinds of data are stored using these formats? Can you give any advice for how you work with these data formats?

- **TSV and CSV** for tables: the entry catalogue (`catalog/entries.tsv`) and each simulation's per-residue analysis (`files/contacts/by_resid_postprocess.csv`). pandas reads both directly.
- **Apache Parquet** for the per-residue table across the whole database (`catalog/residues.parquet`, ~11 million rows). Parquet is columnar and compressed, so you can read only the columns and rows you need without downloading the whole file. pandas, Polars, DuckDB and Amazon Athena all read it, and `pyarrow.dataset` can filter it straight from S3, as below.
- **PDB** (Protein Data Bank format) for molecular structures: the protein as prepared, the final coarse-grained frame (MARTINI beads), and the final frame converted to atomistic detail. Open them with molecular viewers such as [Mol*](https://molstar.org/viewer/), PyMOL or VMD, or with Python libraries such as MDAnalysis, MDTraj or Biopython. The `group_*.contacts.pdb` files keep contact fractions in the temperature-factor (B-factor) column, so colouring by B-factor shows where lipids touch the protein.
- **YAML** for analysis results (`*.latest.yml`), for example bilayer thickness in `thinning.latest.yml`. Read it with PyYAML.
- **ZIP** archives of GROMACS inputs (`files/run/cg.zip`, `at.zip`), to rerun or extend any simulation with [GROMACS](https://www.gromacs.org/).
- **JPEG** images of each system.

For large-scale analysis, start from the two catalogue files. Go to the per-simulation folders when you need structures or finer detail.

### Q: Can you show us an example of downloading and loading data from your dataset?

The entry catalogue is the natural starting point: one row per simulation.

In [ ]:
entries = pd.read_csv(io.BytesIO(read("catalog/entries.tsv")), sep="\t")
print(len(entries), "simulations")
entries.head()

Each simulation's per-residue table says where every residue sits along the membrane normal (`z`, Å from the bilayer centre) and for what fraction of the simulation it touches lipid head groups, lipid tails or water. The columns are listed in the documentation.

In [ ]:
res = pd.read_csv(io.BytesIO(read(f"simulations/{PDB}_default_dppc/files/contacts/by_resid_postprocess.csv")), index_col=0)
res[["chainID", "resSeq", "resName", "DSSP", "z", "group=Head", "group=Tail", "group=Solvent"]].head(10)

The bilayer thickness for this simulation, far from the protein (bulk) and next to it (annular):

In [ ]:
thinning = yaml.safe_load(read(f"simulations/{PDB}_default_dppc/thinning.latest.yml"))
thinning

### Q: A picture is worth a thousand words. Show us a visual (or several!) from your dataset that either illustrates something informative about your dataset, or that you think might excite someone to dig in further.

First, a picture of the assembled system: the lipid surface around ScrY, coloured by thickness (red thinner and blue thicker than the bulk bilayer).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, name in zip(axes, ["front-distortions.jpg", "front-cg_lipids.jpg"]):
    ax.imshow(Image.open(io.BytesIO(read(f"simulations/{PDB}_default_dppc/images/{name}"))))
    ax.set_title(name)
    ax.axis("off")
plt.show()

Next, the same simulation as numbers: each residue of one chain along its sequence, at its depth in the membrane, coloured by how much of the simulation it spends touching lipid tails. The β-strands cross the bilayer, and the residues facing the tails sit between the head-group layers at about ±20 Å.

In [ ]:
chain = res[res["chainID"] == res["chainID"].iloc[0]]
fig, ax = plt.subplots(figsize=(12, 4))
ax.axhspan(-20, 20, color="0.93", zorder=0)
sc = ax.scatter(chain["resSeq"], chain["z"], c=chain["group=Tail"], cmap="viridis", s=14, vmin=0, vmax=1)
ax.set_xlabel("Residue number")
ax.set_ylabel("Depth z (Å from bilayer centre)")
ax.set_title(f"{PDB.upper()} chain {chain['chainID'].iloc[0]}: residue depth and lipid-tail contact")
fig.colorbar(sc, label="Fraction of time touching lipid tails")
plt.show()

Across the whole database: how thick is the membrane next to each protein, compared with the bulk bilayer? Most proteins thin the membrane around them.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(entries["thickness_change_A"].dropna(), bins=80, range=(-15, 10), color="#2f6fd0")
ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel("Annular minus bulk bilayer thickness (Å)")
ax.set_ylabel("Simulations")
ax.set_title("Membrane thinning next to proteins, all MemProtMD simulations")
plt.show()

### Q: What is one question that you have answered using these data? Can you show us how you came to that answer?

**Where in the membrane does each amino acid prefer to sit?** With thousands of structures simulated the same way, MemProtMD gives a large, consistent census of the membrane-protein interface. The classic picture appears directly from the data: hydrophobic residues (Leu, Ile, Phe) fill the bilayer core. Aromatic Trp and Tyr form "belts" at the interfaces near the lipid head groups. Positively charged Arg and Lys reach up to the head groups and out into the water. Their long side chains "snorkel" so that the charge stays out of the hydrophobic core.

We need only three columns of the per-residue Parquet file. `pyarrow.dataset` reads just those columns from S3, not the whole file.

In [ ]:
s3fs = pyarrow.fs.S3FileSystem(anonymous=True, region=REGION)
residues = ds.dataset(f"{BUCKET}/catalog/residues.parquet", filesystem=s3fs, format="parquet")

aas = ["LEU", "PHE", "TRP", "TYR", "ARG", "LYS"]
table = residues.to_table(columns=["resName", "z", "tail"], filter=ds.field("resName").isin(aas))
df = table.to_pandas()
print(f"{len(df):,} residues from {residues.count_rows():,} in the database")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
bins = list(range(-50, 51, 2))
for aa, colour in zip(aas, ["#888888", "#444444", "#e0a526", "#c47f00", "#2f6fd0", "#17946b"]):
    z = df.loc[df["resName"] == aa, "z"]
    ax.hist(z, bins=bins, density=True, histtype="step", lw=2, label=aa, color=colour)
ax.axvspan(-20, 20, color="0.93", zorder=0)
ax.set_xlabel("Depth z (Å from bilayer centre); the shaded band is the bilayer between the phosphates")
ax.set_ylabel("Share of residues")
ax.set_title("Where amino acids sit in the membrane, across all MemProtMD simulations")
ax.legend()
plt.show()

The same question by contact rather than depth: the average fraction of the simulation that each amino acid spends touching lipid tails.

In [ ]:
df.groupby("resName")["tail"].mean().reindex(aas).plot.bar(color="#2f6fd0", figsize=(8, 3), ylabel="Mean tail contact", xlabel="")
plt.show()

### Q: What is one unanswered question that you think could be answered using these data? Do you have any recommendations or advice for someone wanting to answer this question?

**Can we predict which residues of a membrane protein touch lipids, and where specific lipids bind, from structure or sequence alone?**

`residues.parquet` gives per-residue labels for about 9,500 structures, all from the same simulation protocol: contacts with head groups, each tail bead, water and ions, plus depth. That makes it a ready-made training set for machine-learning models of the membrane interface. For example, a graph neural network on the structure, or a protein language model fine-tuned on sequence, could predict lipid contact or membrane depth per residue. Such a model would help place AlphaFold models and designed proteins in membranes, and could highlight candidate lipid binding sites to test with longer or atomistic simulations.

Advice:

- Split training and test sets by protein family (the `uniprot`, `pfam` and `tcdb` columns of `entries.tsv`), not at random. Many proteins have several structures in the PDB, and a random split leaks near-identical examples into the test set.
- Use `instance == 1` to keep one copy of repeated chains.
- These are coarse-grained simulations in a single lipid (DPPC). Contacts are a strong guide to where lipids sit, but specific binding of other lipids (cholesterol, PIP2, cardiolipin) needs simulations in more complex membranes. The GROMACS inputs in `files/run/` are a starting point.